# CUDA PIC backend conformance and timing spike

Choose a **T4 GPU runtime** for correctness, or optionally an **A100** for float64 throughput. Only deterministic synthetic inputs are generated. No token, Drive mount, study data, or production solver installation is used.

Claude must commit and push this spike first. Set the full 40-character **commit containing `spike/`**, then run all cells. A branch name or the pre-spike base commit is insufficient. CUDA kernels compile at first use here; Mac CPU checks do not certify CUDA or Metal execution.

The four operations are periodic CIC/TSC deposition, matched gather, relativistic Boris push, and a periodic discrete-Laplacian FFT Poisson solve. Radiation reaction is explicitly off for these normalized kernel controls. Float64 numerical tolerance is used instead of bitwise equality because FFT implementations and parallel atomic summation orders differ.


In [ ]:
SPIKE_COMMIT = ""  #@param {type:"string"}
RUN_FULL_TIMINGS = True  #@param {type:"boolean"}
CHECK_MLX_CUDA = False  #@param {type:"boolean"}
REPEATS = 3  #@param {type:"integer"}


In [ ]:
import json
import os
from pathlib import Path
import re
import subprocess
import sys

assert re.fullmatch(r"[0-9a-fA-F]{40}", SPIKE_COMMIT), "Set a full pushed spike commit SHA."
assert (3, 12) <= sys.version_info[:2] <= (3, 13), "Use Python 3.12–3.13: pinned NumPy requires 3.12+, and Taichi wheels stop at 3.13."
assert REPEATS >= 1
subprocess.run(["nvidia-smi"], check=True)
ROOT = Path("/content") / ("lw-gpu-spike-" + SPIKE_COMMIT[:12])
RESULTS = Path("/content/lw-gpu-spike-results") / SPIKE_COMMIT[:12]
RESULTS.mkdir(parents=True, exist_ok=True)
if not ROOT.exists():
    subprocess.run(["git", "clone", "--no-checkout", "https://github.com/benfolsom/LW_integrator", str(ROOT)], check=True)
subprocess.run(["git", "fetch", "--depth=1", "origin", SPIKE_COMMIT], cwd=ROOT, check=True)
subprocess.run(["git", "checkout", "--detach", SPIKE_COMMIT], cwd=ROOT, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
assert actual.lower() == SPIKE_COMMIT.lower()
assert (ROOT / "spike/requirements-cuda.txt").is_file(), "This commit does not contain the spike. Ask Claude to push it."
(RESULTS / "commit.txt").write_text(actual + "\n")
ENV = dict(os.environ, PYTHONDONTWRITEBYTECODE="1", OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", LW_SPIKE_CACHE="/content/lw-gpu-spike-cache")

def run_logged(label, command):
    log = RESULTS / (label + ".log")
    with log.open("w") as handle:
        completed = subprocess.run(command, cwd=ROOT, env=ENV, stdout=handle, stderr=subprocess.STDOUT)
    print(log.read_text()[-6000:])
    print("Exit:", completed.returncode, "Log:", log)
    completed.check_returncode()

# Install dependencies in the disposable Colab runtime; no editable checkout install.
run_logged("install", [sys.executable, "-m", "pip", "install", "-r", "spike/requirements-cuda.txt"])
(RESULTS / "pip-freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))


In [ ]:
# These commands are sequential. No more than one compute child runs at once.
run_logged("reference-tests", [sys.executable, "-m", "pytest", "-c", "/dev/null", "spike/test_reference.py", "-q"])
for dtype in ("float64", "float32"):
    for backend in ("cupy", "taichi-cuda"):
        label = backend + "-" + dtype + "-conformance"
        run_logged(label, [sys.executable, "-m", "spike.conformance", "--backends", backend,
                           "--dtype", dtype, "--grids", "16", "64", "128",
                           "--output", str(RESULTS / (label + ".json"))])

# Record the actual CuPy runtime/driver versions as well as nvidia-smi.
CUDA_INFO_CODE = """import json
import cupy as cp
p = cp.cuda.runtime.getDeviceProperties(0)
name = p['name']
print(json.dumps(dict(gpu=name.decode() if isinstance(name, bytes) else name,
                     compute_capability=[p['major'], p['minor']],
                     cuda_runtime=cp.cuda.runtime.runtimeGetVersion(),
                     cuda_driver=cp.cuda.runtime.driverGetVersion(),
                     cupy=cp.__version__), indent=2))
cp.show_config()
"""
run_logged("cuda-runtime", [sys.executable, "-c", CUDA_INFO_CODE])


In [ ]:
if RUN_FULL_TIMINGS:
    for dtype in ("float64", "float32"):
        for backend in ("cupy", "taichi-cuda"):
            label = backend + "-" + dtype + "-timings"
            run_logged(label, [sys.executable, "-m", "spike.benchmark", "--backends", backend,
                               "--dtype", dtype, "--particles", "100000", "1000000", "10000000",
                               "--grids", "64", "128", "--repeats", str(REPEATS),
                               "--output", str(RESULTS / (label + ".json"))])


## Optional MLX CUDA check

This separately tests the documented CUDA 12 build, native FFT, the custom CUDA API, and float64 availability. The PIC MLX adapter uses float32. A working float32 probe alone does not establish a float64 replacement for CuPy. MLX currently documents GPU float64 as unsupported. The CUDA package requires SM 7.5 or newer, driver 550.54.14 or newer, and glibc 2.35 or newer; incompatible runtimes should record a failed probe.

Sources: [MLX installation](https://ml-explore.github.io/mlx/build/html/install.html), [data types](https://ml-explore.github.io/mlx/build/html/python/data_types.html), and [custom CUDA API](https://ml-explore.github.io/mlx/build/html/python/_autosummary/mlx.core.fast.cuda_kernel.html).


In [ ]:
if CHECK_MLX_CUDA:
    run_logged("mlx-install", [sys.executable, "-m", "pip", "install", "mlx[cuda12]==0.32.3"])
    # The probe records unsupported float64 without hiding that result.
    run_logged("mlx-probe", [sys.executable, "-m", "spike.probe_mlx_cuda", "--output", str(RESULTS / "mlx-probe.json")])
    run_logged("mlx-conformance", [sys.executable, "-m", "spike.conformance", "--backends", "mlx-cuda",
                                   "--grids", "16", "64", "128", "--output", str(RESULTS / "mlx-conformance.json")])
    if RUN_FULL_TIMINGS:
        run_logged("mlx-timings", [sys.executable, "-m", "spike.benchmark", "--backends", "mlx-cuda",
                                  "--repeats", str(REPEATS), "--output", str(RESULTS / "mlx-timings.json")])
    (RESULTS / "pip-freeze-with-mlx.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))


In [ ]:
from IPython.display import Markdown, display
lines = ["| Backend | Precision | N | Grid | Shape | Operation | Median ms |", "|---|---|---:|---:|---|---|---:|"]
for path in sorted(RESULTS.glob("*-timings.json")):
    for row in json.loads(path.read_text())["rows"]:
        if row["status"] == "measured":
            lines.append(f"| {row['backend']} | {row['dtype']} | {row['particles']} | {row['grid']}³ | {'CIC' if row['order']==1 else 'TSC'} | {row['kernel']} | {row['median_ms']:.3f} |")
display(Markdown("\n".join(lines)))

# Archive results for download; no upload or external writes.
import tarfile
ARCHIVE = RESULTS.parent / ("cuda-gpu-spike-" + SPIKE_COMMIT[:12] + ".tar.gz")
with tarfile.open(ARCHIVE, "w:gz") as archive:
    archive.add(RESULTS, arcname=RESULTS.name)
print("Download via Colab's Files panel:", ARCHIVE)
